In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "README.md").is_file() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "README.md").is_file():
    raise FileNotFoundError("Run this notebook from the project directory or one of its subdirectories.")


# Train V3 — final preprocessing pipeline

Only the final route is retained. Intermediate data stays in memory; only `train_final_physics_v3.csv` is saved.


## 1. Minimal station-wise preprocessing


In [1]:
import numpy as np
import pandas as pd

# 1. 원본 데이터 로드
df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "train_v2.csv")
df['time'] = pd.to_datetime(df['time'])

def process_station_minimal(group):
    g = group.sort_values('time').set_index('time').copy()
    st = g['station'].iloc[0]
    
    # 기지별 고유 계수
    rayleigh = 1.64 if st == 'I-ORS' else 1.62 if st == 'G-ORS' else 1.63
    gust_f = 1.10 if st == 'I-ORS' else 1.12 if st == 'G-ORS' else 1.15
    tp_f = 4.10 if st == 'I-ORS' else 3.75 if st == 'G-ORS' else 3.60
    sensor_h = 41.0 if st == 'I-ORS' else 31.0 if st == 'G-ORS' else 30.0
    alpha = 0.11 if st == 'I-ORS' else 0.13 if st == 'G-ORS' else 0.14
    h_to_u10 = (10.0 / sensor_h) ** alpha

    # ---------------------------------------------------------
    # 1. 1칸(10분) 징검다리 결측만 선형 보간 (limit=1)
    # ---------------------------------------------------------
    num_cols = ['hs', 'tp', 'hmax', 'wspd', 'gust', 'airt', 'relh', 'caph']
    g[num_cols] = g[num_cols].interpolate(method='time', limit=1)

    for col in ['wdir', 'wvdir']:
        rad = np.deg2rad(g[col])
        u = np.cos(rad).interpolate(method='time', limit=1)
        v = np.sin(rad).interpolate(method='time', limit=1)
        g[col] = (np.rad2deg(np.arctan2(v, u)) % 360.0).round(2)

    # ---------------------------------------------------------
    # 2. 동시간대 물리 상호 교차 대치 (확실한 관계만)
    # ---------------------------------------------------------
    # 풍향 <-> 파향
    g['wdir'] = g['wdir'].fillna(g['wvdir'])
    g['wvdir'] = g['wvdir'].fillna(g['wdir'])

    # 파고 상호 대치 (Rayleigh)
    g['hs'] = g['hs'].fillna(g['hmax'] / rayleigh)
    g['hmax'] = g['hmax'].fillna(g['hs'] * rayleigh)

    # 파고 -> 풍속 추정 (Wilson 역산)
    estimated_u10 = np.sqrt(g['hs'].dropna() / 0.0246)
    estimated_wspd = estimated_u10 / h_to_u10
    g['wspd'] = g['wspd'].fillna(g['gust'] / gust_f)
    g['wspd'] = g['wspd'].fillna(estimated_wspd)
    g['gust'] = g['gust'].fillna(g['wspd'] * gust_f)

    # 풍속 -> 파고 추정 (Wilson 정방향)
    u10 = g['wspd'] * h_to_u10
    g['hs'] = g['hs'].fillna(0.0246 * (u10 ** 2))
    g['hmax'] = g['hmax'].fillna(g['hs'] * rayleigh)

    # 파주기 (JONSWAP)
    g['tp'] = g['tp'].fillna(tp_f * np.sqrt(g['hs'].clip(lower=0.01)))

    return g.reset_index()

# 2. 기지별 전처리 적용 (Pandas 3의 groupby.apply 변경을 피하기 위해 명시적 반복 사용)
processed_stations = [process_station_minimal(group) for _, group in df.groupby('station', sort=False)]
df_clean = pd.concat(processed_stations, ignore_index=True)
print("Stage 1 complete (in memory):", df_clean.shape)


Stage 1 complete (in memory): (236304, 25)


## 2. Station physics imputation


In [2]:
import numpy as np
import pandas as pd

# Stage 1 result is passed in memory; no intermediate CSV is read.
df = df_clean.copy()
df['time'] = pd.to_datetime(df['time'])

class StationPhysicsImputer:
    def __init__(self):
        self.STATION_SPECS = {
            'S-ORS': {'temp_amp': 2.4, 'mean_p': 1015.0, 'gust_factor': 1.15, 'rayleigh': 1.63, 'tp_factor': 3.60},
            'G-ORS': {'temp_amp': 1.6, 'mean_p': 1013.5, 'gust_factor': 1.12, 'rayleigh': 1.62, 'tp_factor': 3.75},
            'I-ORS': {'temp_amp': 1.0, 'mean_p': 1011.8, 'gust_factor': 1.10, 'rayleigh': 1.64, 'tp_factor': 4.10}
        }

    def process_station(self, group: pd.DataFrame, st_name: str) -> pd.DataFrame:
        g = group.sort_values('time').copy()
        specs = self.STATION_SPECS.get(st_name, self.STATION_SPECS['G-ORS'])

        # 시간 보조 변수
        month = g['time'].dt.month
        hour = g['time'].dt.hour + g['time'].dt.minute / 60.0

        # -------------------------------------------------------------
        # 1. 열역학 물리 복원 (기온 airt, 습도 relh)
        # -------------------------------------------------------------
        seasonal_airt = g.groupby(month)['airt'].transform('mean')
        diurnal_cycle = specs['temp_amp'] * np.sin(2 * np.pi * (hour - 8.0) / 24.0)
        g['airt'] = g['airt'].fillna(seasonal_airt + diurnal_cycle).bfill().ffill()

        seasonal_relh = g.groupby(month)['relh'].transform('mean')
        g['relh'] = g['relh'].fillna(seasonal_relh - (diurnal_cycle * 3.5)).clip(15.0, 100.0).bfill().ffill()

        # -------------------------------------------------------------
        # 2. 정역학 기압 복원 (기압 caph)
        # -------------------------------------------------------------
        seasonal_caph = g.groupby(month)['caph'].transform('mean')
        g['caph'] = g['caph'].fillna(seasonal_caph).fillna(specs['mean_p']).bfill().ffill()

        # -------------------------------------------------------------
        # 3. 풍파 물리 평형 복원 (wspd, gust, wdir, hs, tp, hmax, wvdir)
        # -------------------------------------------------------------
        winter_mask = month.isin([11, 12, 1, 2])
        default_wspd = pd.Series(np.where(winter_mask, 8.5, 6.0), index=g.index)
        default_wdir = pd.Series(np.where(winter_mask, 315.0, 180.0), index=g.index)

        g['wspd'] = g['wspd'].fillna(default_wspd).bfill().ffill()
        g['gust'] = g['gust'].fillna(g['wspd'] * specs['gust_factor']).bfill().ffill()
        g['wdir'] = g['wdir'].fillna(default_wdir).bfill().ffill()
        g['wvdir'] = g['wvdir'].fillna(g['wdir']).bfill().ffill()

        # SMB 파랑 에너지 평형식
        u10 = g['wspd'] * 0.86
        smb_hs = (0.243 * (u10 ** 2) / 9.81).clip(lower=0.2)
        
        g['hs'] = g['hs'].fillna(smb_hs).bfill().ffill()
        g['hmax'] = g['hmax'].fillna(g['hs'] * specs['rayleigh']).bfill().ffill()
        g['tp'] = g['tp'].fillna(specs['tp_factor'] * np.sqrt(g['hs'])).bfill().ffill()

        # 물리적 유효범위 클리핑
        g['wspd'] = g['wspd'].clip(0, 45.0)
        g['gust'] = np.maximum(g['gust'].clip(0, 60.0), g['wspd'])
        g['hs'] = g['hs'].clip(0.01, 15.0)
        g['hmax'] = np.maximum(g['hmax'].clip(0.01, 20.0), g['hs'])
        g['tp'] = g['tp'].clip(1.0, 25.0)
        g['wdir'] = g['wdir'] % 360.0
        g['wvdir'] = g['wvdir'] % 360.0

        return g

    def run(self, full_df: pd.DataFrame) -> pd.DataFrame:
        results = []
        # include_groups 인자 제거하여 버전 호환성 확보
        for st_name, group in full_df.groupby('station'):
            g = group.copy()
            g = self.process_station(g, st_name)
            results.append(g)

        res = pd.concat(results, axis=0).sort_values(['station', 'time']).reset_index(drop=True)
        cols = ['station', 'time', 'hs', 'tp', 'hmax', 'wvdir', 'wspd', 'gust', 'wdir', 'airt', 'relh', 'caph']
        return res[cols]


# 2. Station physics imputation (in memory)
print("[2/3] 관측소별 물리 결측치 보간 파이프라인 가동...")
imputer = StationPhysicsImputer()
df_filled = imputer.run(df)

print("Stage 2 complete (in memory):", df_filled.shape)
print(df_filled.isna().sum())


[2/3] 관측소별 물리 결측치 보간 파이프라인 가동...
Stage 2 complete (in memory): (236304, 12)
station    0
time       0
hs         0
tp         0
hmax       0
wvdir      0
wspd       0
gust       0
wdir       0
airt       0
relh       0
caph       0
dtype: int64


## 3. Physics feature generation and final save


In [3]:
import numpy as np
import pandas as pd

PHYSICS_PATH = PROJECT_ROOT / "data" / "processed" / "train_final_physics_v4.csv"


def add_physics_features(group: pd.DataFrame) -> pd.DataFrame:
    """
    기존 물리 변수 + 파랑 동역학(첨예도/에너지) + 고파랑 모멘텀 파생변수 확장
    """
    g = group.sort_values("time").copy()

    # -------------------------------------------------------------
    # 1. 기존 바람/기압 파생변수
    # -------------------------------------------------------------
    g["wspd_mean_6h"] = g["wspd"].rolling(36, min_periods=12).mean()
    g["wspd_mean_12h"] = g["wspd"].rolling(72, min_periods=24).mean()
    g["gust_max_6h"] = g["gust"].rolling(36, min_periods=12).max()
    g["gust_max_12h"] = g["gust"].rolling(72, min_periods=24).max()
    g["gust_minus_wspd"] = g["gust"] - g["wspd"]

    # 풍향-파향 일치도
    angle_difference = np.deg2rad(g["wdir"] - g["wvdir"])
    g["wind_wave_alignment"] = np.cos(angle_difference)
    g["wind_wave_diff"] = np.abs((g["wdir"] - g["wvdir"] + 180) % 360 - 180)

    # 기압 변화율 (단기 3h + 중기 6h + 장기 12h)
    g["caph_change_3h"] = g["caph"] - g["caph"].shift(18)
    g["caph_change_6h"] = g["caph"] - g["caph"].shift(36)
    g["caph_change_12h"] = g["caph"] - g["caph"].shift(72)

    # -------------------------------------------------------------
    # 2. [추가] 파고 모멘텀 및 누적 에너지 (단기/중기 예측 핵심)
    # -------------------------------------------------------------
    # 파고 변화 속도 (1시간, 3시간 차분)
    g["hs_diff_1h"] = g["hs"] - g["hs"].shift(6)
    g["hs_diff_3h"] = g["hs"] - g["hs"].shift(18)

    # 파고 이동 통계량
    g["hs_mean_6h"] = g["hs"].rolling(36, min_periods=12).mean()
    g["hs_mean_12h"] = g["hs"].rolling(72, min_periods=24).mean()
    g["hs_max_6h"] = g["hs"].rolling(36, min_periods=12).max()
    g["hs_max_12h"] = g["hs"].rolling(72, min_periods=24).max()

    # -------------------------------------------------------------
    # 3. [추가] 파랑 동역학 (Steepness, Energy, Forcing)
    # -------------------------------------------------------------
    # 파장 근사 L = 1.56 * Tp^2 (m)
    wavelength = 1.56 * (g["tp"] ** 2)
    # 파랑 첨예도 (H/L) -> 너울(<0.02) vs 풍파(>0.033) 판별 지표
    g["wave_steepness"] = g["hs"] / np.maximum(wavelength, 1.0)

    # 파랑 에너지 (E ~ Hs^2)
    g["wave_energy"] = g["hs"] ** 2

    # 유효 순풍 에너지 (파도를 직접 키우는 바람 힘 = U^2 * cos(theta))
    g["effective_wind_forcing"] = (g["wspd"] ** 2) * g["wind_wave_alignment"]

    # -------------------------------------------------------------
    # 4. [추가] 2차원 파랑 직교 벡터 분해
    # -------------------------------------------------------------
    wave_radians = np.deg2rad(g["wvdir"])
    g["u_wave"] = g["hs"] * np.sin(wave_radians)
    g["v_wave"] = g["hs"] * np.cos(wave_radians)

    return g


def create_physics_train() -> pd.DataFrame:
    """새로운 물리 파생변수를 결합하여 train_final_physics.csv 생성"""
    print("[1/2] 물리 파생변수 생성 중...")
    # Stage 2 result is passed in memory; this is the only saved output.
    df = df_filled.copy()
    df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values(["station", "time"]).reset_index(drop=True)

    # 기지별 롤링 및 물리 계산
    physics_list = []
    for _, group in df.groupby("station"):
        physics_list.append(add_physics_features(group))

    physics = pd.concat(physics_list, ignore_index=True)
    physics = physics.sort_values(["station", "time"]).reset_index(drop=True)

    # 바람 직교 벡터
    wind_radians = np.deg2rad(physics["wdir"])
    physics["u_wind"] = physics["wspd"] * np.sin(wind_radians)
    physics["v_wind"] = physics["wspd"] * np.cos(wind_radians)

    # Rolling/lag features are undefined only at each station's first rows.
    # Fill those boundary values inside the same station so model inputs stay finite.
    feature_columns = [column for column in physics.columns if column not in ["station", "time"]]
    physics[feature_columns] = physics.groupby("station")[feature_columns].transform(
        lambda group: group.bfill().ffill()
    )

    # 파일 저장
    physics.to_csv(PHYSICS_PATH, index=False)
    print(f"[2/2] '{PHYSICS_PATH}' 저장 완료! Shape: {physics.shape}")
    print(f"생성된 전체 컬럼 목록:\n{physics.columns.tolist()}")
    return physics


if __name__ == "__main__":
    create_physics_train()

[1/2] 물리 파생변수 생성 중...
[2/2] 'train_final_physics_v4.csv' 저장 완료! Shape: (236304, 35)
생성된 전체 컬럼 목록:
['station', 'time', 'hs', 'tp', 'hmax', 'wvdir', 'wspd', 'gust', 'wdir', 'airt', 'relh', 'caph', 'wspd_mean_6h', 'wspd_mean_12h', 'gust_max_6h', 'gust_max_12h', 'gust_minus_wspd', 'wind_wave_alignment', 'wind_wave_diff', 'caph_change_3h', 'caph_change_6h', 'caph_change_12h', 'hs_diff_1h', 'hs_diff_3h', 'hs_mean_6h', 'hs_mean_12h', 'hs_max_6h', 'hs_max_12h', 'wave_steepness', 'wave_energy', 'effective_wind_forcing', 'u_wave', 'v_wave', 'u_wind', 'v_wind']


In [ ]:
# ============================================================
# 5. EXTENDED CAUSAL PHYSICS FEATURES FOR EXP14
# ============================================================
import numpy as np
import pandas as pd

OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "train_final_physics_v4.csv"
physics = pd.read_csv(OUTPUT_PATH, parse_dates=["time"])

def add_extended_features(group):
    g = group.sort_values("time").copy()

    # Wave state and variability. Full windows prevent partial-history leakage.
    g["hmax_hs_ratio"] = g["hmax"] / g["hs"].clip(lower=0.01)
    g["wave_energy_flux"] = (g["hs"] ** 2) * g["tp"]
    g["hs_std_6h"] = g["hs"].rolling(36, min_periods=36).std()
    g["hs_std_12h"] = g["hs"].rolling(72, min_periods=72).std()
    g["hs_range_6h"] = g["hs"].rolling(36, min_periods=36).max() - g["hs"].rolling(36, min_periods=36).min()
    g["hs_range_12h"] = g["hs"].rolling(72, min_periods=72).max() - g["hs"].rolling(72, min_periods=72).min()

    # Wind evolution and the wind component perpendicular to wave travel.
    g["wspd_diff_1h"] = g["wspd"] - g["wspd"].shift(6)
    g["wspd_diff_3h"] = g["wspd"] - g["wspd"].shift(18)
    g["wspd_std_6h"] = g["wspd"].rolling(36, min_periods=36).std()
    g["wspd_std_12h"] = g["wspd"].rolling(72, min_periods=72).std()
    g["wind_cross_wave"] = g["wspd"] * np.sin(np.deg2rad(g["wdir"] - g["wvdir"]))

    # Atmospheric state: Magnus dew point and a trailing 24-hour pressure anomaly.
    rh_fraction = g["relh"].clip(lower=1e-3, upper=100.0) / 100.0
    gamma = np.log(rh_fraction) + (17.625 * g["airt"]) / (243.04 + g["airt"])
    g["dew_point"] = (243.04 * gamma) / (17.625 - gamma)
    g["caph_anomaly_24h"] = g["caph"] - g["caph"].rolling(144, min_periods=144).mean()

    # Deterministic calendar cycles are available at inference time.
    hour = g["time"].dt.hour + g["time"].dt.minute / 60.0
    day_of_year = g["time"].dt.dayofyear
    g["hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
    g["hour_cos"] = np.cos(2 * np.pi * hour / 24.0)
    g["doy_sin"] = np.sin(2 * np.pi * day_of_year / 365.25)
    g["doy_cos"] = np.cos(2 * np.pi * day_of_year / 365.25)
    return g.replace([np.inf, -np.inf], np.nan)

physics = pd.concat(
    [add_extended_features(group) for _, group in physics.groupby("station", sort=True)],
    ignore_index=True,
)
physics = physics.sort_values(["station", "time"]).reset_index(drop=True)
physics.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

new_features = [
    "hmax_hs_ratio", "wave_energy_flux", "hs_std_6h", "hs_std_12h",
    "hs_range_6h", "hs_range_12h", "wspd_diff_1h", "wspd_diff_3h",
    "wspd_std_6h", "wspd_std_12h", "wind_cross_wave", "dew_point",
    "caph_anomaly_24h", "hour_sin", "hour_cos", "doy_sin", "doy_cos",
]
print(f"Extended V4 features saved: {OUTPUT_PATH}")
print(pd.DataFrame({"missing": physics[new_features].isna().sum()}).T)
